In [1]:
import pandas as pd
import numpy as np

# Load the original dataset
original_data = pd.read_csv("main_dataset.csv")

# Separate data based on blockchain usage
no_blockchain_data = original_data[original_data["IS_BLOCKCHAIN"] == "no"]
with_blockchain_data = original_data[original_data["IS_BLOCKCHAIN"] == "yes"]

# Calculate means and standard deviations
means_no_bc = no_blockchain_data.mean(numeric_only=True)
stds_no_bc = no_blockchain_data.std(numeric_only=True)
means_with_bc = with_blockchain_data.mean(numeric_only=True)
stds_with_bc = with_blockchain_data.std(numeric_only=True)

# Number of samples
n_samples = 5000

# Non-Blockchain Scenario
data_no_bc = pd.DataFrame({
    "Lead_Time_days": np.random.triangular(1, means_no_bc["Lead_Time_days"] * 1.10, 35, n_samples),
    "Inventory_Accuracy": np.random.normal(means_no_bc["Inventory_Accuracy"] * 0.95, stds_no_bc["Inventory_Accuracy"] * 1.1, n_samples),
    "Supplier_Lead_Time_Variability_days": np.random.triangular(0, means_no_bc["Supplier_Lead_Time_Variability_days"] * 1.20, 12, n_samples),
    "Inventory_Turnover_Ratio": np.random.normal(means_no_bc["Inventory_Turnover_Ratio"] * 0.95, stds_no_bc["Inventory_Turnover_Ratio"], n_samples),
    "Customer_Satisfaction": np.random.normal(means_no_bc["Customer_Satisfaction"] * 0.93, stds_no_bc["Customer_Satisfaction"], n_samples),
    "Supplier_Relationship_Score": np.random.normal(means_no_bc["Supplier_Relationship_Score"] * 0.95, stds_no_bc["Supplier_Relationship_Score"], n_samples),
    "Order_Fulfillment_Rate": np.random.normal(means_no_bc["Order_Fulfillment_Rate"] * 0.93, stds_no_bc["Order_Fulfillment_Rate"], n_samples),
    "Operational_Efficiency_Score": np.random.normal(means_no_bc["Operational_Efficiency_Score"] * 0.90, stds_no_bc["Operational_Efficiency_Score"], n_samples)
})

# Blockchain Scenario (Further increase variance)
data_with_bc = pd.DataFrame({
    "Lead_Time_days": np.random.triangular(1, means_with_bc["Lead_Time_days"] * 0.80, 35, n_samples),  # Widened max (30 → 35)
    "Inventory_Accuracy": np.random.normal(means_with_bc["Inventory_Accuracy"] * 1.15, stds_with_bc["Inventory_Accuracy"] * 0.9, n_samples),
    "Supplier_Lead_Time_Variability_days": np.random.triangular(0, means_with_bc["Supplier_Lead_Time_Variability_days"] * 0.80, 15, n_samples),  # Widened max (10 → 15)
    "Inventory_Turnover_Ratio": np.random.normal(means_with_bc["Inventory_Turnover_Ratio"] * 1.10, stds_with_bc["Inventory_Turnover_Ratio"], n_samples),
    "Customer_Satisfaction": np.random.normal(means_with_bc["Customer_Satisfaction"] * 1.10, stds_with_bc["Customer_Satisfaction"], n_samples),
    "Supplier_Relationship_Score": np.random.normal(means_with_bc["Supplier_Relationship_Score"] * 1.10, stds_with_bc["Supplier_Relationship_Score"], n_samples),
    "Order_Fulfillment_Rate": np.random.normal(means_with_bc["Order_Fulfillment_Rate"] * 1.10, stds_with_bc["Order_Fulfillment_Rate"], n_samples),
    "Operational_Efficiency_Score": np.random.normal(means_with_bc["Operational_Efficiency_Score"] * 1.15, stds_with_bc["Operational_Efficiency_Score"], n_samples)
})

# Composite functions
def compute_transparency(df):
    return (0.3 * df["Customer_Satisfaction"] + 0.3 * df["Order_Fulfillment_Rate"] +
            0.2 * df["Operational_Efficiency_Score"] + 0.1 * df["Inventory_Accuracy"] +
            0.1 * df["Supplier_Relationship_Score"])

def compute_traceability(df):
    df["Lead_Time_days_norm"] = (df["Lead_Time_days"] - df["Lead_Time_days"].min()) / (df["Lead_Time_days"].max() - df["Lead_Time_days"].min())
    df["Supplier_Lead_Time_Variability_days_norm"] = (df["Supplier_Lead_Time_Variability_days"] - df["Supplier_Lead_Time_Variability_days"].min()) / (df["Supplier_Lead_Time_Variability_days"].max() - df["Supplier_Lead_Time_Variability_days"].min())
    df["Inventory_Turnover_Ratio_norm"] = (df["Inventory_Turnover_Ratio"] - df["Inventory_Turnover_Ratio"].min()) / (df["Inventory_Turnover_Ratio"].max() - df["Inventory_Turnover_Ratio"].min())
    noise = np.random.normal(0, 0.07, len(df))  # Increased noise (0.05 → 0.1)
    traceability = (0.35 * (1 - df["Lead_Time_days_norm"]) + 0.35 * (1 - df["Supplier_Lead_Time_Variability_days_norm"]) +
                   0.3 * df["Inventory_Turnover_Ratio_norm"]) + noise
    return np.clip(traceability, 0, 1)

# Apply composite calculations
for df in [data_no_bc, data_with_bc]:
    df["Transparency"] = compute_transparency(df)
    df["Traceability"] = compute_traceability(df)

# Markov Chain with adjusted risk calculation
base_risk_no_bc = -0.4 * data_no_bc["Transparency"] - 0.4 * data_no_bc["Traceability"]
base_risk_with_bc = -0.4 * data_with_bc["Transparency"] - 0.6 * data_with_bc["Traceability"]  # Increased Traceability weight (0.5 → 0.6)

transition_matrix_no_bc = np.array([
    [0.70, 0.20, 0.10],
    [0.25, 0.50, 0.25],
    [0.10, 0.30, 0.60]
])

transition_matrix_with_bc = np.array([
    [0.85, 0.10, 0.05],
    [0.40, 0.50, 0.10],
    [0.20, 0.50, 0.30]
])

def simulate_markov_chain(transition_matrix, n_steps, initial_state=0):
    states = [initial_state]
    for _ in range(n_steps - 1):
        current_state = states[-1]
        next_state = np.random.choice([0, 1, 2], p=transition_matrix[current_state])
        states.append(next_state)
    return states

state_to_risk = {0: -0.3, 1: 0.0, 2: 0.3}

risk_states_no_bc = simulate_markov_chain(transition_matrix_no_bc, n_samples)
risk_states_with_bc = simulate_markov_chain(transition_matrix_with_bc, n_samples)

data_no_bc["Supply_Chain_Risk"] = base_risk_no_bc + [state_to_risk[state] for state in risk_states_no_bc]
data_with_bc["Supply_Chain_Risk"] = base_risk_with_bc + [state_to_risk[state] for state in risk_states_with_bc]

# Save datasets
data_no_bc.to_csv("synthetic_data_no_blockchain_final_traceability.csv", index=False)
data_with_bc.to_csv("synthetic_data_with_blockchain_final_traceability.csv", index=False)

print("Final synthetic data with enhanced Traceability generated and saved.")

Final synthetic data with enhanced Traceability generated and saved.
